In [43]:
import time
import copy
from collections import deque
from ortools.sat.python import cp_model

In [45]:
def read_puzzles(file):
    with open(file, 'r') as f:
        return [line.strip() for line in f if line.strip()]

def string_to_grid(puzzle_str):
    return [[int(puzzle_str[i * 9 + j]) for j in range(9)] for i in range(9)]

def grid_to_string(grid):
    return ''.join(str(cell) for row in grid for cell in row)

In [53]:
def write_solutions_to_file(filepath, solutions):
    with open(filepath, 'w') as f:
        for sol in solutions:
            f.write(sol + "\n")

def compute_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                candidates = set(range(1, 10))
                for k in range(9):
                    candidates.discard(board[r][k])  # row
                    candidates.discard(board[k][c])  # col
                br, bc = 3 * (r // 3), 3 * (c // 3)
                for i in range(3):
                    for j in range(3):
                        candidates.discard(board[br + i][bc + j])
                domains[(r, c)] = candidates
    return domains

# --- AC-3 Constraint Propagation ---
def find_related_cells(pos):
    r, c = pos
    related = set((r, i) for i in range(9) if i != c)
    related.update((i, c) for i in range(9) if i != r)
    box_r, box_c = 3 * (r // 3), 3 * (c // 3)
    for i in range(3):
        for j in range(3):
            cell = (box_r + i, box_c + j)
            if cell != pos:
                related.add(cell)
    return related

def revise(domains, xi, xj):
    removed = set()
    for x in domains[xi]:
        if all(x == y for y in domains[xj]):
            removed.add(x)
    if removed:
        domains[xi] -= removed
        return True
    return False

def ac3(domains):
    queue = deque((a, b) for a in domains for b in find_related_cells(a) if b in domains)
    while queue:
        xi, xj = queue.popleft()
        if xi not in domains or xj not in domains:
            continue
        if revise(domains, xi, xj):
            if not domains[xi]:
                return False
            for neighbor in find_related_cells(xi):
                if neighbor != xj and neighbor in domains:
                    queue.append((neighbor, xi))
    return True

# --- Backtracking Search ---
def choose_variable(domains):
    #Select variable with the fewest options (MRV heuristic)
    return min(domains, key=lambda k: len(domains[k]))

def recursive_backtrack(board, domains):
    if not domains:
        return True  # All variables assigned

    var = choose_variable(domains)
    for val in sorted(domains[var]):
        board_copy = copy.deepcopy(board)
        domain_copy = copy.deepcopy(domains)

        board_copy[var[0]][var[1]] = val
        del domain_copy[var]

        # Update neighbors' domains
        for neighbor in find_related_cells(var):
            if neighbor in domain_copy:
                domain_copy[neighbor].discard(val)

        if ac3(domain_copy):
            if recursive_backtrack(board_copy, domain_copy):
                board[:] = board_copy
                return True

    return False  # No valid assignment found

def solve_single_board(board):
    domains = compute_domains(board)
    if not ac3(domains):
        return None
    if recursive_backtrack(board, domains):
        return board
    return None

def solve_all_boards(input_file, output_file):
    raw_puzzles = load_puzzles_from_file(input_file)
    results = []
    for puzzle_str in raw_puzzles:
        board = string_to_grid(puzzle_str)
        solution = solve_single_board(board)
        if solution:
            results.append(grid_to_string(solution))
        else:
            results.append("No solution")
    write_solutions_to_file(output_file, results)
input_file = "input.txt"
output_file = "output.txt"
solve_all_boards(input_file, output_file)

In [55]:
# ------ GOOGLE ORTOOL CODE------
from ortools.sat.python import cp_model

def solve_sudoku_ortools(grid):
    model = cp_model.CpModel()
    cells = [[model.NewIntVar(1, 9, f"cell_{r}_{c}") for c in range(9)] for r in range(9)]
    for row in range(9):
        for col in range(9):
            if grid[row][col] != 0:
                model.Add(cells[row][col] == grid[row][col])
    for i in range(9):
        model.AddAllDifferent(cells[i])  # Row constraint
        model.AddAllDifferent([cells[r][i] for r in range(9)])  # Column constraint
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            block = [cells[r][c] for r in range(br, br+3) for c in range(bc, bc+3)]
            model.AddAllDifferent(block)
    solver = cp_model.CpSolver()
    status = solver.Solve(model)
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return [[solver.Value(cells[r][c]) for c in range(9)] for r in range(9)]
    else:
        return None


In [57]:
#----- GPT Solution ------
def is_valid(grid, row, col, num):
    for i in range(9):
        if grid[row][i] == num or grid[i][col] == num:
            return False
    start_row, start_col = 3*(row//3), 3*(col//3)
    for i in range(3):
        for j in range(3):
            if grid[start_row + i][start_col + j] == num:
                return False
    return True

def solve_basic(grid):
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                for num in range(1, 10):
                    if is_valid(grid, r, c, num):
                        grid[r][c] = num
                        if solve_basic(grid):
                            return True
                        grid[r][c] = 0
                return False
    return True

In [63]:
# ---- comparison-----
def compare_solvers(input_file):
    puzzles = read_puzzles(input_file)
    total_basic, total_ortools, total_csp = 0, 0, 0
    all_equal = True
    for idx, puzzle in enumerate(puzzles):
        print(f"\nPuzzle {idx + 1}:")
        grid = string_to_grid(puzzle)

        # --- GPT Solver ---
        basic_grid = copy.deepcopy(grid)
        start = time.time()
        solve_basic(basic_grid)
        total_basic += time.time() - start
        basic_str = grid_to_string(basic_grid)

        # --- OR-Tools ---
        ort_grid = string_to_grid(puzzle)
        start = time.time()
        solved_ort = solve_sudoku_ortools(ort_grid)
        total_ortools += time.time() - start
        ort_str = grid_to_string(solved_ort) if solved_ort else "No solution"

        # --- Mine solution ---
        csp_grid = string_to_grid(puzzle)
        start = time.time()
        solved_csp = solve_single_board(csp_grid)
        total_csp += time.time() - start
        csp_str = grid_to_string(solved_csp) if solved_csp else "No solution"

        # Print one-line comparison
        print("GPT == ORTools:", basic_str == ort_str)
        print("ORTools == AC3+Backtracking:", ort_str == csp_str)
        print("AC3+Backtrackin == GPT:", csp_str == basic_str)

        if not (basic_str == ort_str == csp_str):
            all_equal = False
            print("Inconsistency Found!")

    print("\n=== Summary ===")
    print(f"GPT Solver Total Time: {total_basic:.4f} sec")
    print(f"OR-Tools Solver Total Time: {total_ortools:.4f} sec")
    print(f"AC3 + BACKTRACKING Solver Total Time: {total_csp:.4f} sec")
    if all_equal:
        print("All solvers produced identical results.")
    else:
        print("At least one puzzle had inconsistent outputs.")

# --- Run Comparison ---
compare_solvers("input.txt")


Puzzle 1:
GPT == ORTools: True
ORTools == AC3+Backtracking: True
AC3+Backtrackin == GPT: True

=== Summary ===
GPT Solver Total Time: 0.0046 sec
OR-Tools Solver Total Time: 0.0307 sec
AC3 + BACKTRACKING Solver Total Time: 0.0153 sec
All solvers produced identical results.
